# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from groq import Groq

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'

groq = Groq(api_key=os.getenv("GROQ_API_KEY"))
api_model = os.getenv("GROQ_API_MODEL")
# openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [12]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [18]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [19]:
select_relevant_links("https://edwarddonner.com")

NameError: name 'openai' is not defined

In [24]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {api_model}")
    response = groq.chat.completions.create(
        model=api_model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print("Found {len(links['links'])} relevant links")
    return links

In [25]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [26]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Edge0/Edge0-35B-A3B-preview
Updated
about 21 hours ago
•
37.1k
•
3.31k
deepseek-ai/DeepSeek-V4.1-Flash
Updated
8 days ago
•
391k
•
3.01k
Qwen/Qwen3.8-27B
Updated
Aug 14
•
7.46M
•
15.5k
m-a-p/YuE2-3B
Updated
1 day ago
•
11.6k
•
723
TokenRhythm/NeoHorse-1-4B


In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [31]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [32]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nEdge0/Edge0-35B-A3B-preview\nUpdated\nabout 21 hours ago\n•\n37.1k\n•\n3.31k\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdat

In [33]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=api_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


**Hugging Face – The AI Community Building the Future**  

---

## 🌟 Company Overview  

Hugging Face is the world’s leading collaboration platform for the machine‑learning community. From research labs to Fortune‑500 enterprises, millions of developers, data scientists, and AI enthusiasts rely on the Hub to create, share, and deploy models, datasets, and applications at scale.  

- **Founded:** 2016 (Paris & New York)  
- **Mission:** Democratise AI by providing open‑source tools and a vibrant community where anyone can build and ship intelligent systems.  
- **Core Products:**  
  - **Models** – 2 M+ ready‑to‑use models spanning NLP, vision, audio, multimodal, and reinforcement learning.  
  - **Datasets** – 500 k+ curated datasets for training, evaluation, and benchmarking.  
  - **Spaces** – Interactive, shareable AI apps (1 M+ applications) that run on zero‑cost compute.  
  - **Buckets** – Scalable storage for model artefacts, fine‑tuning results, and data pipelines.  
  - **Enterprise Solutions** – PRO, Inference Endpoints, Inference Providers, and dedicated support for large‑scale deployments.  

---

## 🚀 Platform Highlights  

| Feature | What It Does | Why It Matters |
|---------|--------------|----------------|
| **Model Hub** | Browse, download, and contribute 2 M+ models (e.g., Qwen, DeepSeek, Edge0). | Accelerates research and product development; reduces time‑to‑value. |
| **Spaces** | Host live demos, chatbots, music generators, video editors—all with a single click. | Turns prototypes into shareable experiences for stakeholders and users. |
| **Datasets Library** | Search and download high‑quality datasets (GLUE, IMDB, SQuAD, UltraData). | Provides the data foundation for robust AI training and evaluation. |
| **Inference Endpoints** | Managed, low‑latency API serving for any model on the Hub. | Enables production‑grade scaling without infra overhead. |
| **Enterprise PRO** | Private repositories, SSO, audit logs, and priority support. | Gives businesses the security and governance they need. |
| **Community‑Driven Open‑Source** | Transformers, Diffusers, Tokenizers, and many more libraries with >100 k contributors. | Keeps the ecosystem cutting‑edge and transparent. |

---

## 🤝 Community & Culture  

- **Open Collaboration:** Over 106 k followers on the Hub, active Discord, forums, and weekly “Daily Papers” summarising the latest research.  
- **Transparency:** All code, models, and datasets are publicly visible; telemetry is optional and community‑driven.  
- **Inclusivity:** Diverse global community—contributors span 150+ countries, speaking 50+ languages.  
- **Innovation at Speed:** Weekly “Trending” models and “Featured” Spaces showcase rapid breakthroughs.  
- **Learning First:** Hugging Face Fundamentals on DataCamp, extensive docs, tutorials, and a “Learn” portal for every skill level.  

> *“Hugging Face is the collaboration platform for the machine learning community.”* – Brand Statement  

---

## 📈 Customers & Impact  

| Segment | Typical Use Cases | Notable Examples |
|---------|-------------------|------------------|
| **Start‑ups & Developers** | Rapid prototyping, AI‑powered SaaS features | AI chatbots, content generation tools |
| **Enterprises** | Secure model governance, scalable inference | Finance risk models, healthcare diagnostics |
| **Research Labs & Universities** | Open‑source reproducibility, dataset sharing | Academic papers, benchmark leaderboards |
| **Product Teams** | Embedding pretrained models into apps | Voice assistants, recommendation engines |

The platform powers **thousands of production services** daily, handling billions of inference calls worldwide.

---

## 💼 Careers & Opportunities  

Hugging Face’s talent philosophy mirrors its product ethos: **open, collaborative, and impact‑driven**.  

- **Roles:** Engineering (ML, infra, frontend), Research, Product, Community & Partnerships, Sales & Enterprise Success.  
- **Work Environment:** Remote‑first, flexible hours, and a culture that rewards curiosity and contribution to open‑source.  
- **Growth:** Employees work alongside world‑renowned researchers, contribute to industry‑shaping libraries, and see their work deployed at scale instantly.  

> *If you love building tools that millions of people use every day, Hugging Face is where your ideas can shape the future of AI.*  

---

## 📞 Get In Touch  

- **Website:** https://huggingface.co  
- **Community:** Discord, Forum, GitHub, Twitter  
- **Enterprise Inquiries:** Use the “Enterprise” menu for demos and pricing.  
- **Career Page:** Search “Hugging Face jobs” for the latest openings.  

---

*Join the world’s most vibrant AI community and help build the future—one model, dataset, and Space at a time.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [36]:
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model=api_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [37]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


**Hugging Face – The AI Community Building the Future**  
*Collaboration platform for machine learning models, datasets, and applications.*

---

### 🚀 Company Overview  
- **Mission:** Empower the global machine‑learning community to create, discover, and collaborate on AI resources faster and more openly.  
- **Core Offering:** A unified hub where developers, researchers, and enterprises can host unlimited public **models**, **datasets**, **Spaces** (interactive apps), and **Buckets** (storage).  
- **Scale:**  
  - **2 M+** open‑source models (including cutting‑edge LLMs such as Qwen, DeepSeek, Edge0).  
  - **500 k+** datasets (GLUE, IMDB, SQuAD, UltraData, etc.).  
  - **1 M+** live AI applications (video generation, music synthesis, source‑grounded notes).  

---

### 🌟 Platform Highlights  

| Feature | What It Does | Why It Matters |
|--------|--------------|----------------|
| **Model Hub** | Browse, download, and fine‑tune over 2 M models. | Accelerates research and product development. |
| **Datasets** | Access curated, version‑controlled datasets. | Guarantees reproducibility and data quality. |
| **Spaces** | Deploy interactive demos with zero‑code or custom code. | Turns research into usable products instantly. |
| **Inference Endpoints & Providers** | Managed, scalable inference for production workloads. | Reduces ops overhead for enterprises. |
| **Enterprise & PRO Plans** | Dedicated support, private repos, compliance tools. | Meets corporate security and SLA requirements. |
| **Community Tools** | Discord, Forum, GitHub, Daily Papers, Blog. | Keeps the ecosystem vibrant and up‑to‑date. |

---

### 🤝 Community & Culture  

- **Open‑source first:** All core assets are freely available; contributions are encouraged from anyone.  
- **Collaborative spirit:** The hub functions as a social network for ML engineers—think “GitHub for AI.”  
- **Transparency:** Public model cards, dataset licenses, and community‑driven documentation.  
- **Diversity & Inclusion:** A global community spanning academia, startups, and Fortune‑500s, reflected in multilingual support and inclusive branding (bright HF colors, friendly logo).  
- **Learning & Growth:** Regular “Learn” resources, daily papers summaries, and a thriving Discord channel for real‑time knowledge exchange.  

---

### 📊 Customers & Use Cases  

- **Enterprises:** Leverage private hubs, inference endpoints, and enterprise support for production‑grade AI (e.g., fintech risk modeling, e‑commerce recommendation, health‑care diagnostics).  
- **Startups & SaaS:** Rapidly prototype and launch AI‑powered features using Spaces and pre‑trained models.  
- **Research & Academia:** Share reproducible experiments via public models/datasets; cite source‑grounded AI notes.  
- **Creative Industries:** Generate video, music, and multimedia content with community‑built LoRAs and generative models.  

---

### 👩‍💼 Careers & Opportunities  

Hugging Face is constantly expanding its talent pool. Typical roles include:  

- **Machine Learning Engineers & Researchers** – building next‑gen models and improving inference pipelines.  
- **Software Engineers (Backend, Frontend, DevOps)** – scaling the Hub, improving UX, and ensuring reliability.  
- **Product & Growth** – shaping the roadmap for PRO/Enterprise offerings and community programs.  
- **Community & Developer Relations** – fostering partnerships, running events, and supporting open‑source contributors.  

*Why join?* Work at the intersection of cutting‑edge AI and open‑source culture, with a remote‑first policy, competitive benefits, and a mission to democratize intelligence.

---

### 📣 Get Involved  

1. **Explore:** Visit the Hub, try out top‑trending models or Spaces.  
2. **Contribute:** Fork a model, submit a dataset, or improve documentation on GitHub.  
3. **Connect:** Join the Discord, Forum, or attend community meet‑ups.  
4. **Upgrade:** For businesses, explore **Hugging Face PRO** or **Enterprise Support** for private, secure deployments.  

---

### 📞 Contact & Resources  

- **Website:** https://huggingface.co  
- **Docs & API:** Docs, Inference Endpoints, Storage Buckets  
- **Community Channels:** Discord, Forum, GitHub, Blog, Daily Papers  
- **Brand Assets:** Official logos (SVG/PNG/AI) and color palette (#FFD21E, #FF9D00, #6B7280) for partners and marketers.  

*Join the AI community that’s building the future—today.*

In [38]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("GROQ platform", "https://groq.com")

Selecting relevant links for https://groq.com by calling openai/gpt-oss-120b
Found {len(links['links'])} relevant links


**Groq – The Premier Neocloud for Fast AI Inference**  
*Turning AI training into real‑world value at speed, scale, and affordability.*

---

## Why Groq?

- **Purpose‑Built Inference Cloud** – Our LPX architecture runs alongside NVIDIA’s next‑gen GPUs, delivering unparalleled throughput with the reliability of bare‑metal infrastructure.  
- **No Trade‑off Between Speed & Cost** – Hundreds of megawatts of capacity are already online, and more are being added, so you get ultra‑fast latency without premium pricing.  
- **Eliminating the Bottleneck** – As AI workloads grow, inference becomes the limiting factor. Groq was engineered from silicon to the cloud to keep pace with the most demanding models.  

> “Training creates the possibility. Inference creates the value.” – *Adam Winter, CEO*

---

## Who’s Using Groq?

- **5 M+ developers** and **thousands of AI‑native companies** run **trillions of tokens** on Groq every week.  
- Global enterprises across **North America, Europe, Middle East, and APAC** rely on our platform for production‑ready AI services.  
- Partners such as **NVIDIA** (Cloud Partner, technology licensing) amplify our ecosystem and accelerate customer time‑to‑value.  

*Case‑in‑point*: Early adopters of the **NVIDIA Groq‑3 LPX** and **Vera Rubin NVL72** hardware have reported up to **2× faster inference** with half the energy consumption.

---

## Investor Highlights

- **$350 M Series A (2026)** – Led to the launch of the world’s leading AI inference cloud.  
- **$650 M additional raise (June 2026)** – Funding dedicated to scaling infrastructure and expanding global data‑center footprint.  
- **Strategic Partnerships** – NVIDIA Cloud Partner status and non‑exclusive licensing agreements position Groq at the forefront of hardware‑software co‑innovation.  

*Our growth trajectory*: From a silicon‑first LPU to a fully integrated neocloud, we are building a **global, hyperscale inference platform** that is both **capital‑efficient** and **enterprise‑grade**.

---

## Culture & People

- **Innovation‑First** – Engineers and leaders who have taken the LPU from prototype to production, merging deep silicon expertise with cloud operations.  
- **Customer‑Obsessed** – Every commit, product, and agent task is measured against real‑world inference performance.  
- **Global Mindset** – With data centers spanning four continents, we foster a collaborative, multicultural environment that values speed, reliability, and sustainability.  

### Leadership Team

| Role | Name | Background |
|------|------|------------|
| **CEO** | **Adam Winter** | 30‑year tech veteran; former Cisco, cloud & AI entrepreneur; joined Groq 2024, CEO 2026 |
| **CFO** | **Matt Eng** | Finance & ops leader from VMware, Pivotal, EMC; drives capital efficiency |
| **COO** | **Alan Rice** | (Details pending – operational excellence in scaling global infrastructure) |

---

## Careers – Join the Inference Revolution

- **Roles in demand**: Hardware engineering, cloud infrastructure, AI software, product management, and enterprise sales.  
- **What we offer**: Competitive compensation, equity participation, and the chance to shape the next generation of AI infrastructure.  
- **Why Groq?**: Work alongside industry pioneers, impact billions of AI requests daily, and help eliminate the inference bottleneck for the world’s biggest AI workloads.

*Ready to accelerate AI?* Visit **[Groq Careers]** (link) to explore open positions and apply.

---

## Get Started Today

- **Start building** – Spin up a Groq inference node in minutes via the GroqCloud portal.  
- **Contact us** – For demos, pricing, or partnership inquiries, reach out through the **Contact** page.  

*Fast, affordable, and scalable inference is no longer a dream. It’s Groq.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>